# 01 — Using a Real Quantum Computer

This notebook hands you the keys to an actual IBM quantum processor: you take a quantity you have so far computed *exactly* on the simulator and re-run it, unchanged, on real hardware.

**Prerequisites:** `01/11_shot_noise_and_first_circuit`, `01/12_from_states_to_density_matrix`.

**What you'll be able to do**
- Get an IBM Quantum account and store your credentials safely, outside the repository.
- Explain why a QPU is driven through *primitives* (`SamplerV2`/`EstimatorV2`), not `backend.run`.
- Transpile a circuit to an ISA circuit and read the resulting measurement counts.
- Use the `USE_HARDWARE` switch that every notebook in this module relies on.

Welcome to real hardware. Until now the course has been simulator-first, and for good reason: the simulator is exact, offline, and reproducible, which made it the perfect place to *understand* the mathematics. From here, the very same code reaches out to a superconducting chip cooled to a few millikelvin in an IBM datacentre. You have done the hard conceptual work; this module is the applied coda where you watch those ideas survive contact with a physical device. The switch stays `USE_HARDWARE = False` by default, so everything in this notebook runs on your laptop right now — and flips to a real QPU the moment your credentials are in place.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2
from qot_course.hardware.runtime import select_backend

np.random.seed(42)
viz.use_course_style()

## 1 · Getting an account and storing your credentials

Running on IBM hardware starts with a free account on the [IBM Quantum Platform](https://quantum.ibm.com). Once you sign in, your dashboard shows an **API token** — a long secret string that authenticates you to the runtime service.

Treat that token like a password. There are two safe ways to give it to Qiskit, and one rule that overrides both: **never hardcode a token in a notebook and never commit it to git.**

- **Recommended — `save_account`.** Run `QiskitRuntimeService.save_account(...)` once, locally. It writes the token to `~/.qiskit/` (your home directory, outside this repository), so every run-cell afterwards needs no token at all and nothing secret ever touches the project tree.
- **Alternative — a git-ignored `.env`.** Put `IBM_QUANTUM_TOKEN=...` in a `.env` at the repo root. That file is already in `.gitignore`, and `select_backend` reads the token from the environment automatically.

The next cell shows both recipes. It does not contact IBM and is safe to run as-is — the actual token lives outside the notebook.

In [ ]:
# Two ways to give your token to QiskitRuntimeService (run ONE, once, locally).
# A) Recommended -- saves to ~/.qiskit/ (outside the repo); run-cells need no token:
#    from qiskit_ibm_runtime import QiskitRuntimeService
#    QiskitRuntimeService.save_account(
#        channel="ibm_quantum_platform",   # current platform (legacy "ibm_quantum" retired mid-2025)
#        token="<YOUR_TOKEN from your IBM Quantum Platform dashboard>",
#        overwrite=True,
#    )
# B) A git-ignored .env at the repo root (already in .gitignore):
#        IBM_QUANTUM_TOKEN=<YOUR_TOKEN>
#    (export it / load it; select_backend then uses it automatically.)
print("Credentials are configured outside this notebook -- nothing secret lives here.")

## 2 · Why primitives, not `backend.run`

On today's IBM Quantum Platform a QPU is reached only through the runtime **primitives** — the two high-level entry points that wrap a real device:

- **`SamplerV2`** samples bitstrings from your circuit and returns **counts** (how many times each outcome appeared). This is the primitive you use whenever you need a probability distribution over measurement results.
- **`EstimatorV2`** returns **expectation values** of observables directly, handling the measurement bases for you.

Both run inside an optional `Session` or batch that groups your jobs together. The local `backend.run` you met in `01/15` still works, but only for simulators — it is not the door to real hardware.

One step is mandatory before any QPU run: **transpilation to an ISA circuit.** "ISA" stands for the device's *instruction set architecture* — its native gate set (for IBM chips, gates such as `rz`, `sx`, and `cx`) wired across its specific qubit connectivity. Your abstract circuit, written with `h` and `cx` on logical qubits, is rewritten into those native instructions on physical qubits. `generate_preset_pass_manager(backend=...)` builds the transpiler that performs this rewrite.

In [ ]:
USE_HARDWARE = False  # flip to True once your credentials are set (see section 1)
backend, label, is_real = select_backend(use_hardware=USE_HARDWARE)
print(f"Running on: {label}  (real hardware = {is_real})")

qc = QuantumCircuit(2, 2)
qc.h(0); qc.cx(0, 1)         # a Bell pair
qc.measure([0, 1], [0, 1])

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa = pm.run(qc)             # transpile to the backend's ISA
print("ISA depth:", isa.depth(), "| ISA gates:", dict(isa.count_ops()))

counts = SamplerV2(mode=backend).run([isa], shots=4096).result()[0].data.c.get_counts()
print("counts:", counts)

In [ ]:
viz.plot_counts(counts)
plt.show()

**Read the figure.** A perfect Bell pair only ever yields `00` or `11`, the two correlated outcomes. Yet the bars show small `01` and `10` populations too: these are outcomes an ideal Bell pair forbids, and they appear here because the backend carries a real device's read-out and gate errors. With the default `USE_HARDWARE=False` those errors come from the *modeled* FakeManila noise — a calibrated approximation, not a live measurement; flip `USE_HARDWARE=True` and the same code reports the literal device's errors instead. Notice also what transpilation did to the circuit: the printout above reports the ISA depth and a native gate set (`rz`, `sx`, `cx`) rather than the `h`/`cx` you wrote — the same logical circuit, re-expressed in the hardware's own language.

## 3 · Sessions, the queue, and cost

A few practicalities once you point at real hardware. `QiskitRuntimeService().least_busy(...)` picks an available QPU for you, so you need not name a specific chip (the `select_backend` helper used in this notebook calls `least_busy` for you, so the switch cell handles QPU selection automatically). Real devices are shared, so your job **enters a queue** and runs when its turn comes; the number of **shots** you request translates directly into device runtime, which is what you are billed for. Wrapping several jobs in a `Session` or batch lets them run back-to-back and cuts the queueing overhead between them.

Finally, a pointer rather than a lesson: the primitives expose **error mitigation** through *resilience levels*, techniques that reduce the bias of noisy results. We name it here so you know the lever exists; this module shows the raw device first, and revisits mitigation only where it changes a conclusion.

## Your turn

- Flip `USE_HARDWARE = True` (after setting your credentials in section 1) and re-run the switch cell. Compare the printed `label` with what you saw offline — it should now name a real QPU.
- Raise the shot count to `20000` in the `SamplerV2(...).run(...)` call and re-run. Watch the fraction of forbidden `01`/`10` outcomes settle toward a stable value as the statistics improve.
- Print `backend.num_qubits` and `isa.layout` to see how large the device is and which physical qubits the transpiler chose for your two-qubit circuit.

## What you built

- A reproducible on-ramp to real quantum hardware: the `USE_HARDWARE` switch that runs offline today and reaches an IBM QPU unchanged once credentials are set.
- A safe credential workflow — `save_account` to `~/.qiskit/` or a git-ignored `.env` — with no secret anywhere in the notebook.
- A first primitive run end-to-end: build a Bell pair, transpile it to an ISA circuit, sample it with `SamplerV2`, and read the counts.
- An eye for the device's fingerprint: forbidden outcomes and a transpiled native gate set that the ideal simulator never shows.

## What's next

Next you re-run the **Born rule** on real hardware, watching how read-out and gate errors bend the measured probabilities away from the textbook values you derived in module 01.

## References

- Qiskit IBM Runtime documentation — primitives, transpilation, and sessions: <https://docs.quantum.ibm.com>.
- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 1–2 (Cambridge University Press, 2010).

**Previous:** `notebooks/04_QuantumOptimalTransport/20_frontier_synthesis.ipynb` · **Next:** `notebooks/05_RealQuantumHardware/02_born_rule_on_real_hardware.ipynb`